In [ ]:
# Prerequisites (Notebook → Settings, right sidebar):
#   1. Internet = ON  (required for git clone + pip)
#   2. Accelerator = GPU (T4 recommended)
#   3. Input dataset = LLVIP (wait until 100% downloaded before running)

import os
REPO = '/kaggle/working/microghost'
BRANCH = 'Ibtida'  # use 'main' for default branch

if os.path.isdir(os.path.join(REPO, '.git')):
    print('Repo already cloned — pulling latest...')
    %cd {REPO}
    !git fetch origin
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    !git clone -b {BRANCH} https://github.com/Sayjad21/microghost.git {REPO}
    %cd {REPO}

In [ ]:
import os, subprocess, sys
REPO = '/kaggle/working/microghost'
os.chdir(REPO)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'torch', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/cu121', '-q',
], check=True)
print('✅ Dependencies installed in', os.getcwd())

In [ ]:
import os

# Override manually if auto-detect fails (your Kaggle path example):
# MANUAL_LLVIP = '/kaggle/input/datasets/saisumathappala/llvip-dataset/LLVIP'
MANUAL_LLVIP = None

def find_llvip_root(base='/kaggle/input'):
    """Walk /kaggle/input and find folder with infrared/ + Annotations/."""
    if not os.path.isdir(base):
        return None
    for root, dirs, _files in os.walk(base):
        lower = {d.lower() for d in dirs}
        if 'infrared' in lower and 'annotations' in lower:
            return root
    return None

base_input = '/kaggle/input'
llvip_path = MANUAL_LLVIP or find_llvip_root(base_input)

print('Top-level mounts:')
if os.path.isdir(base_input):
    for item in os.listdir(base_input):
        print(f'  /kaggle/input/{item}')
else:
    print('  (none)')

if llvip_path and os.path.isdir(llvip_path):
    print(f'\n✅ LLVIP root: {llvip_path}')
    print(f'   train images: {len(os.listdir(os.path.join(llvip_path, "infrared", "train")))}')
else:
    llvip_path = None
    print('\n❌ LLVIP not found — add dataset in sidebar, wait for 100% download, re-run.')

In [ ]:
import os, subprocess, sys

REPO = '/kaggle/working/microghost'
if not llvip_path:
    raise RuntimeError('LLVIP not found — run the dataset cell above first.')
if not os.path.isfile(os.path.join(REPO, 'main.py')):
    raise RuntimeError('Repo missing — run Step 0 (clone) first with Internet ON.')

os.chdir(REPO)
print('Training from:', os.getcwd())
print('Dataset:', llvip_path)

subprocess.run([
    sys.executable, 'main.py', 'train',
    '--dataset', 'llvip',
    '--data-root', llvip_path,
    '--batch-size', '32',
    '--num-workers', '2',
    '--epochs', '50',
], check=True)

print('\n✅ Training complete → checkpoints/best_microghost_thermal_v2.pth')

In [ ]:
# Optional: compare v1 vs v2 (needs v1 checkpoint uploaded to checkpoints/)
import os, subprocess, sys

REPO = '/kaggle/working/microghost'
os.chdir(REPO)
v1 = 'checkpoints/best_microghost_thermal_v1.pth'

if os.path.isfile(v1) and llvip_path:
    subprocess.run([
        sys.executable, 'compare_models.py',
        '--dataset', 'llvip', '--data-root', llvip_path,
    ], check=True)
else:
    print('Skipping compare_models (no v1 checkpoint). Use v2 for inference.')